<a href="https://colab.research.google.com/github/Urvish04/credit-fairness-audit/blob/main/credit_fairness_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- SECTION 1: Data Acquisition ---
# We're using the German Credit dataset (credit-g) from OpenML.
# It's a standard benchmark in algorithmic fairness literature.

import pandas as pd
from sklearn.datasets import fetch_openml

print("Pulling down the credit-g dataset from OpenML... this might take a few seconds.")

# Fetching the dataset (parser='auto' is used to avoid future version warnings)
raw_data = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
df = raw_data.frame

# Quick sanity check: Did we get all the rows we expected?
print(f"\nData loaded successfully! Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

# Let's check our target variable balance before doing anything else.
# We expect around 700 'good' and 300 'bad' risk profiles.
print("\nTarget distribution (Good vs. Bad credit risk):")
print(df['class'].value_counts())

Pulling down the credit-g dataset from OpenML... this might take a few seconds.

Data loaded successfully! Shape: 1000 rows, 21 columns.

Target distribution (Good vs. Bad credit risk):
class
good    700
bad     300
Name: count, dtype: int64


In [2]:
# --- SECTION 2: Feature Preparation & Protected Attribute Extraction ---
# We must separate the protected attributes (sex, age, foreign worker status)
# BEFORE training. The goal is to see if the model proxies these attributes
# through other features (like employment length or housing).

# 1. Extract 'sex' from the 'personal_status' column
# personal_status contains values like 'male single' or 'female div/dep/mar'
df['sex'] = df['personal_status'].apply(lambda x: 'female' if 'female' in str(x).lower() else 'male')

# 2. Binarize 'age' into 'age_group'
# In fairness literature for this dataset, < 25 is conventionally the unprivileged 'young' group
df['age_group'] = df['age'].apply(lambda x: 'young' if x < 25 else 'old')

print("--- Protected Group Counts in the Full Dataset ---")
print("Sex:\n", df['sex'].value_counts().to_string())
print("\nAge Group:\n", df['age_group'].value_counts().to_string())
print("\nForeign Worker:\n", df['foreign_worker'].value_counts().to_string())

# 3. Isolate the target variable (1 for good credit, 0 for bad)
y = df['class'].apply(lambda x: 1 if x == 'good' else 0)

# 4. Drop the protected attributes and the target from our training features
# We also drop the raw 'age' and 'personal_status' columns to prevent data leakage
cols_to_drop = ['class', 'personal_status', 'sex', 'age_group', 'age', 'foreign_worker']
X_raw = df.drop(columns=cols_to_drop)

# 5. One-hot encode the remaining categorical features
# drop_first=True prevents multicollinearity (the dummy variable trap)
X = pd.get_dummies(X_raw, drop_first=True)

print(f"\nFeature prep complete! Training matrix shape: {X.shape}")

--- Protected Group Counts in the Full Dataset ---
Sex:
 sex
male      690
female    310

Age Group:
 age_group
old      851
young    149

Foreign Worker:
 foreign_worker
yes    963
no      37

Feature prep complete! Training matrix shape: (1000, 43)


In [3]:
# --- SECTION 3: Model Training ---
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("Splitting data into 70% training and 30% testing...")

# We stratify by 'y' to ensure the 70/30 split maintains the same ratio of good/bad credit.
# CRITICAL: We pass the original indices (df.index) into the split.
# Why? Because we need to know exactly WHICH rows ended up in the test set
# so we can reattach their protected attributes (sex/age) later for the audit.
indices = df.index
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, indices, test_size=0.3, stratify=y, random_state=42
)

print("Training a baseline Logistic Regression model (blind to protected attributes)...")

# We use max_iter=1000 to ensure the solver converges.
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Generate predictions on the test set
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Training Complete!")
print(f"Baseline Accuracy: {accuracy:.2%}")

Splitting data into 70% training and 30% testing...
Training a baseline Logistic Regression model (blind to protected attributes)...

Model Training Complete!
Baseline Accuracy: 74.67%


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:
# --- SECTION 4: The Fairness Audit (The Core Deliverable) ---
# Now we analyze the predictions on the 30% holdout set.
# We reattach the protected attributes (which the model never saw during training)
# to see if the model's outcomes unfairly disadvantage specific groups.

# Rebuild a dataframe for just the test set results
audit_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred': y_pred,
    'sex': df.loc[idx_test, 'sex'],
    'age_group': df.loc[idx_test, 'age_group']
})

def compute_fairness_metrics(group_col):
    """Calculates Approval Rate and False Negative Rate for each subgroup."""
    results = []
    groups = audit_df[group_col].unique()

    for g in groups:
        subset = audit_df[audit_df[group_col] == g]
        n_total = len(subset)

        # Approval Rate: Percentage of this group predicted as 'good credit' (1)
        approval_rate = subset['y_pred'].mean()

        # False Negative Rate (FNR): Of the actually 'good' profiles, how many were denied?
        actual_goods = subset[subset['y_true'] == 1]
        fnr = 1 - actual_goods['y_pred'].mean() if len(actual_goods) > 0 else None

        results.append({
            'Group': g,
            'Test_Set_Size (N)': n_total,
            'Approval_Rate': round(approval_rate, 4),
            'False_Negative_Rate': round(fnr, 4) if fnr is not None else None
        })

    # Format as a clean table
    return pd.DataFrame(results).set_index('Group').sort_index()

print("--- FAIRNESS AUDIT: SEX ---")
print(compute_fairness_metrics('sex'))

print("\n--- FAIRNESS AUDIT: AGE GROUP ---")
print(compute_fairness_metrics('age_group'))

--- FAIRNESS AUDIT: SEX ---
        Test_Set_Size (N)  Approval_Rate  False_Negative_Rate
Group                                                        
female                 96         0.7396               0.1429
male                  204         0.7794               0.1293

--- FAIRNESS AUDIT: AGE GROUP ---
       Test_Set_Size (N)  Approval_Rate  False_Negative_Rate
Group                                                       
old                  253         0.7747               0.1250
young                 47         0.7234               0.1923


## 5. Disparate Impact Analysis & Regulatory Context

**Calculated Metrics:**
* **Sex (Female vs. Male):** The approval rate for female applicants is 73.96%, compared to 77.94% for male applicants.
    * Disparate Impact Ratio (DIR) = 0.7396 / 0.7794 = **0.95**
* **Age Group (Young vs. Old):** The approval rate for young applicants (<25) is 72.34%, compared to 77.47% for older applicants.
    * Disparate Impact Ratio (DIR) = 0.7234 / 0.7747 = **0.93**

**Interpretation & Conclusion:**
Both ratios (0.95 and 0.93) fall above the US EEOC "4/5ths rule" threshold of 0.80. Therefore, the baseline Logistic Regression model—when explicitly blinded to protected attributes during training—does not exhibit significant adverse impact against the unprivileged groups in this sample. The model demonstrates fairness across these dimensions.

*Data Limitation Note:* The test set contained only 47 "young" applicants. Because this subgroup size ($N=47$) is relatively small, the corresponding disparate impact ratio is sensitive to minor prediction changes and should be monitored over a larger sample size in a production environment.

**Real-World Regulatory Relevance:**
Evaluating models for algorithmic bias is a live compliance requirement. Recently, the Reserve Bank of India (RBI) raised concerns regarding algorithmic lending bias in its draft guidelines for digital lending and the broader discussions around the 'Framework for Responsible and Ethical Enablement of Artificial Intelligence' (FREE AI). Regulatory bodies are increasingly requiring that models be validated for equity to ensure historical data does not systematically disadvantage protected demographics. This audit demonstrates a technical methodology to comply with such regulatory scrutiny by quantifying disparate impact prior to model deployment.

In [5]:
# --- SECTION 5: Extension - Model Comparison ---
# Does a more complex, non-linear model introduce more bias?
# Let's train a Random Forest and compare its fairness to our Logistic Regression.

from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Add RF predictions to our existing audit dataframe
audit_df['rf_y_pred'] = rf_model.predict(X_test)

# Let's check the approval rates for Sex using the new model
rf_approval_by_sex = audit_df.groupby('sex')['rf_y_pred'].mean()
print("\n--- RANDOM FOREST: APPROVAL RATES BY SEX ---")
print(rf_approval_by_sex.round(4).to_string())

# And for Age Group
rf_approval_by_age = audit_df.groupby('age_group')['rf_y_pred'].mean()
print("\n--- RANDOM FOREST: APPROVAL RATES BY AGE GROUP ---")
print(rf_approval_by_age.round(4).to_string())

Training Random Forest Classifier...

--- RANDOM FOREST: APPROVAL RATES BY SEX ---
sex
female    0.8542
male      0.8382

--- RANDOM FOREST: APPROVAL RATES BY AGE GROUP ---
age_group
old      0.8577
young    0.7660


In [6]:
!pip install fairlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 13.2 MB/s eta 0:00:00


In [7]:
# --- SECTION 6: Formal Verification with Fairlearn ---
from fairlearn.metrics import demographic_parity_difference

print("--- FAIRLEARN: DEMOGRAPHIC PARITY DIFFERENCE ---")
print("This measures the absolute difference in approval rates between groups.")
print("A value of 0 means perfect parity.\n")

# Calculate for Sex
dp_diff_sex = demographic_parity_difference(
    audit_df['y_true'],
    audit_df['y_pred'],
    sensitive_features=audit_df['sex']
)
print(f"Demographic Parity Difference (Sex): {dp_diff_sex:.4f}")

# Calculate for Age Group
dp_diff_age = demographic_parity_difference(
    audit_df['y_true'],
    audit_df['y_pred'],
    sensitive_features=audit_df['age_group']
)
print(f"Demographic Parity Difference (Age Group): {dp_diff_age:.4f}")

--- FAIRLEARN: DEMOGRAPHIC PARITY DIFFERENCE ---
This measures the absolute difference in approval rates between groups.
A value of 0 means perfect parity.

Demographic Parity Difference (Sex): 0.0398
Demographic Parity Difference (Age Group): 0.0513
